# Deep Learning - Notebook 21
## Natural Language Processing (NLP) Foundations & Embeddings

Welcome to Module 21.

In Notebook 20, we learned how Convolutional Neural Networks process visual data (image tensors). Now, we turn to another massive domain of Artificial Intelligence: **Natural Language Processing (NLP)**.

Computers do not understand human language, grammar, sarcasm, or context. They only understand numbers and matrices.

To build language AI, we must convert raw unstructured text (emails, customer reviews, support tickets, medical notes) into mathematical vector representations.

In this notebook, we master text preprocessing, TF-IDF vectorization, topic modeling, word embeddings (Word2Vec, GloVe, FastText), Named Entity Recognition (NER), and sentiment classification using an anchor scenario:
* **Automated Customer Support Ticket Router & Sentiment Analyzer:** Building an enterprise NLP engine that receives raw customer support tickets, cleans the text, extracts key entities (names, order IDs, dollar amounts), determines customer sentiment (Angry vs Satisfied), and automatically routes tickets to the correct department (Billing, Technical Support, Returns).

For every topic, we answer three questions:
* What is this in plain English?
* Why are we learning this?
* How is this used in Artificial Intelligence?

---
## Section 1: Text Preprocessing & Cleaning Pipelines

### 1. What is this in plain English?
Raw human text is noisy. People type with typos, random capital letters, punctuation, and extra spaces. Before feeding text to an AI model, we must clean and normalize it:

1. **Tokenization:** Chopping a full sentence into individual word tokens.
   * `"The server crashed!"` $\to$ `["The", "server", "crashed"]`
2. **Stop Word Removal:** Stripping out common filler words that carry no unique semantic signal (e.g., `"the"`, `"is"`, `"at"`, `"which"`).
3. **Normalization (Lowercasing & Punctuation Removal):** Converting all text to lowercase so `"CRASH"`, `"Crash"`, and `"crash"` are treated as the exact same word token.
4. **Stemming vs. Lemmatization:**
   * **Stemming (heuristic slicing):** Crude chopping of word suffixes. (`"running"`, `"runs"`, `"ran"` $\to$ `"run"`).
   * **Lemmatization (vocabulary dictionary lookup):** Reducing words to their true canonical dictionary root form (`"better"` $\to$ `"good"`; `"was"` $\to$ `"be"`).

### 2. Why are we learning this?
Text normalization reduces vocabulary size by 70%, preventing data sparsity and accelerating model training.

### 3. How is this used in AI?
* Cleaning text before feature vectorization in search engines, spam filters, and LLM tokenizers.

In [1]:
import numpy as np
import pandas as pd
import re

# SAMPLE CUSTOMER SUPPORT TICKETS
raw_support_tickets = [
    "URGENT: My billing payment of $150.00 FAILED on 2026-03-10! Please refund immediately.",
    "The new software update is crashing my computer every time I open the dashboard...",
    "I want to RETURN my order #98765. The item arrived damaged and broken.",
    "Great service! The customer support team resolved my issue very quickly."
]

def clean_text_pipeline(text):
    # Step 1: Convert to lowercase
    text = text.lower()
    
    # Step 2: Remove punctuation and special symbols using Regex
    text = re.sub(r"[^\w\s]", "", text)
    
    # Step 3: Tokenize by splitting on whitespace
    tokens = text.split()
    
    # Step 4: Remove English Stop Words (common filler words)
    stop_words = {"my", "of", "on", "please", "the", "is", "my", "every", "time", "i", "to", "and", "a", "very"}
    clean_tokens = [word for word in tokens if word not in stop_words]
    
    return clean_tokens

print("--- Text Cleaning & Tokenization Pipeline ---")
for idx, ticket in enumerate(raw_support_tickets, start=1):
    tokens = clean_text_pipeline(ticket)
    print(f"Ticket {idx} Tokens: {tokens}")

--- Text Cleaning & Tokenization Pipeline ---
Ticket 1 Tokens: ['urgent', 'billing', 'payment', '15000', 'failed', '20260310', 'refund', 'immediately']
Ticket 2 Tokens: ['new', 'software', 'update', 'crashing', 'computer', 'open', 'dashboard']
Ticket 3 Tokens: ['want', 'return', 'order', '98765', 'item', 'arrived', 'damaged', 'broken']
Ticket 4 Tokens: ['great', 'service', 'customer', 'support', 'team', 'resolved', 'issue', 'quickly']


---
## Section 2: Statistical NLP & Vectorization
### Bag-of-Words, TF-IDF, & Topic Modeling (LDA)

### 1. What is this in plain English?
* **Bag-of-Words (Count Vectorizer):** Converts text into a matrix by counting how many times each word appears in a document. (Ignores word order completely!).
* **TF-IDF (Term Frequency - Inverse Document Frequency):** Count Vectorizer has a major flaw: words that appear everywhere (like `"computer"` or `"service"`) get high scores, even though they don't help differentiate specific ticket types.
  * **TF (Term Frequency):** How often a word appears in *this specific document*.
  * **IDF (Inverse Document Frequency):** Penalizes words that appear across *all documents* in the corpus.
  $$\text{TF-IDF}(w, d) = \text{TF}(w, d) \times \log\left(\frac{N}{\text{DF}(w)}\right)$$
  *Result:* Rare, highly informative words (like `"refund"`, `"crash"`, `"damaged"`) get high TF-IDF scores!
* **Topic Modeling (LDA - Latent Dirichlet Allocation):** An unsupervised algorithm that reads 10,000 unlabeled text documents and automatically discovers hidden themes/topics (e.g., Cluster 1 = Billing Issues; Cluster 2 = Hardware Crashes).

### 2. How is this used in AI?
* Vectorizing text for machine learning models (Logistic Regression, Naive Bayes, XGBoost).
* Unsupervised customer feedback categorization.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 1. TF-IDF VECTORIZATION
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(raw_support_tickets)

feature_words = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF Matrix Shape (4 Tickets x Unique Words):", tfidf_matrix.shape)
print("Extracted Feature Words (First 10):", feature_words[:10])

# Displaying top TF-IDF score for Ticket 1
ticket_1_vector = tfidf_matrix[0].toarray().flatten()
top_word_indices = ticket_1_vector.argsort()[::-1][:4]

print("\nTicket 1 Top TF-IDF Keywords:")
for idx in top_word_indices:
    print(f" - Word: '{feature_words[idx]:<12}' | TF-IDF Score: {ticket_1_vector[idx]:.4f}")

# 2. TOPIC MODELING VIA LATENT DIRICHLET ALLOCATION (LDA)
# Discovering 2 hidden topics across support tickets
lda_model = LatentDirichletAllocation(n_components=2, random_state=42)
lda_model.fit(tfidf_matrix)

print("\n--- Discovered Topics via LDA ---")
for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [feature_words[i] for i in topic.argsort()[:-5:-1]]
    print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

TF-IDF Matrix Shape (4 Tickets x Unique Words): (4, 35)
Extracted Feature Words (First 10): ['00' '03' '10' '150' '2026' '98765' 'arrived' 'billing' 'broken'
 'computer']

Ticket 1 Top TF-IDF Keywords:
 - Word: 'urgent      ' | TF-IDF Score: 0.3015
 - Word: 'payment     ' | TF-IDF Score: 0.3015
 - Word: 'refund      ' | TF-IDF Score: 0.3015
 - Word: 'billing     ' | TF-IDF Score: 0.3015

--- Discovered Topics via LDA ---
Topic 1: resolved, great, support, team
Topic 2: crashing, computer, new, software


---
## Section 3: Dense Semantic Word Embeddings
### Word2Vec, GloVe, and FastText

### 1. What is this in plain English?
TF-IDF vectors have a massive limitation: they are **Sparse** (99% zeros) and treat every word as completely independent.
To TF-IDF, the words `"King"` and `"Queen"` are as unrelated as `"King"` and `"Banana"`.

**Word Embeddings** project words into a dense $N$-dimensional vector space (e.g., 100 or 300 dimensions) where **distance represents meaning**.
* Words used in similar contexts sit near each other in vector space.
* They capture semantic analogies mathematically:
  $$\text{Vector("King")} - \text{Vector("Man")} + \text{Vector("Woman")} \approx \text{Vector("Queen")}$$

### 2. The Embedding Architectures:
1. **Word2Vec (Google):** Trains a shallow neural network to predict context words (**CBOW - Continuous Bag of Words**) or predict surrounding words given a target word (**Skip-Gram**).
2. **GloVe (Global Vectors - Stanford):** Factorizes a global word co-occurrence matrix across the entire Wikipedia/web corpus.
3. **FastText (Meta/Facebook):** Breaks words down into character n-grams (e.g., `"apple"` $\to$ `"app"`, `"ppl"`, `"ple"`). This allows FastText to calculate embeddings for out-of-vocabulary words and typos (e.g., `"unrefundability"`)!

### 3. How is this used in AI?
* Initializing embedding layers in Recurrent Neural Networks (RNNs), LSTMs, and Transformers.

In [10]:
# WORD2VEC EMBEDDINGS DEMONSTRATION WITH GENSIM (WITH FALLBACK)

try:
    from gensim.models import Word2Vec

    # Tokenized corpus for training Word2Vec
    tokenized_corpus = [clean_text_pipeline(ticket) for ticket in raw_support_tickets]

    # Train Word2Vec Model
    w2v_model = Word2Vec(
        sentences=tokenized_corpus,
        vector_size=50,   # 50-dimensional dense vector space
        window=3,        # Context window size
        min_count=1,     # Include all words
        workers=1,
        seed=42
    )

    # Get vector for word 'refund'
    word_vector = w2v_model.wv["refund"]
    print("Word2Vec Vector for 'refund' (Shape: 50 dimensions):")
    print(np.round(word_vector[:10], 4), "... [truncated]")

    # Find most similar words
    similar_words = w2v_model.wv.most_similar("refund", topn=2)
    print("\nMost Similar Words to 'refund':", similar_words)

except ImportError:
    print("Note: 'gensim' package not installed. Word2Vec vector math demonstrated mathematically.")

Note: 'gensim' package not installed. Word2Vec vector math demonstrated mathematically.


---
## Section 4: Structural Language Understanding
### Named Entity Recognition (NER) & POS Tagging

### 1. What is this in plain English?
* **Part-of-Speech (POS) Tagging:** Assigning grammatical labels to every word in a sentence (e.g., Noun, Verb, Adjective, Adverb).
* **Named Entity Recognition (NER):** Automatically identifying and extracting real-world entities from unstructured text:
  * **PERSON:** Customer Name (`"Alice Smith"`)
  * **ORG:** Company/Vendor Name (`"FinTech Corp"`)
  * **MONEY:** Monetary Amounts (`"$150.00"`)
  * **DATE:** Dates & Timestamps (`"2026-03-10"`)
  * **CARDINAL:** Order / Tracking Numbers (`"#98765"`)

### 2. Why are we learning this?
NER allows automated systems to route tickets instantly. If a ticket contains a `MONEY` entity and words like `"failed"`, it is automatically routed to the **Billing Department**.

### 3. How is this used in AI?
* Automated document parsing, legal contract auditing, and clinical medical record extraction.

In [3]:
# NAMED ENTITY RECOGNITION (NER) & REGEX ENTITY EXTRACTION

def extract_entities_regex(ticket_text):
    """
    Extracts key operational entities from raw support tickets using Regular Expressions.
    """
    entities = {}
    
    # 1. Extract Monetary Amounts
    money_match = re.findall(r"\$\d+(?:\.\d{2})?", ticket_text)
    entities["MONEY"] = money_match if money_match else None
    
    # 2. Extract Dates (YYYY-MM-DD)
    date_match = re.findall(r"\d{4}-\d{2}-\d{2}", ticket_text)
    entities["DATE"] = date_match if date_match else None
    
    # 3. Extract Order/Tracking Numbers (#12345)
    order_match = re.findall(r"#\d+", ticket_text)
    entities["ORDER_ID"] = order_match if order_match else None
    
    return entities

print("--- Automated Entity Extraction Engine ---")
for idx, ticket in enumerate(raw_support_tickets, start=1):
    extracted = extract_entities_regex(ticket)
    print(f"Ticket {idx}: {extracted}")

--- Automated Entity Extraction Engine ---
Ticket 1: {'MONEY': ['$150.00'], 'DATE': ['2026-03-10'], 'ORDER_ID': None}
Ticket 2: {'MONEY': None, 'DATE': None, 'ORDER_ID': None}
Ticket 3: {'MONEY': None, 'DATE': None, 'ORDER_ID': ['#98765']}
Ticket 4: {'MONEY': None, 'DATE': None, 'ORDER_ID': None}


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# TRAINING DATASET FOR SUPPORT TICKET ROUTING
training_tickets = [
    "Payment failed and overcharged my credit card",
    "Refund my money for transaction failure",
    "App crashes on launch and freezes my screen",
    "Software error code 500 when loading dashboard",
    "Item arrived broken and I want to return it",
    "Wrong size delivered, need an exchange or return",
    "Great service and helpful customer support",
    "Thanks for fixing my account so quickly"
]

department_labels = [
    "Billing", "Billing",
    "Technical", "Technical",
    "Returns", "Returns",
    "General", "General"
]

# BUILD END-TO-END NLP CLASSIFICATION PIPELINE
nlp_router_pipeline = make_pipeline(
    TfidfVectorizer(stop_words="english"),
    LogisticRegression()
)

nlp_router_pipeline.fit(training_tickets, department_labels)

# TEST PIPELINE ON NEW UNSEEN TICKETS
new_incoming_tickets = [
    "I was charged twice on my credit card statement!",
    "The application keeps shutting down on Windows 11"
]

predictions = nlp_router_pipeline.predict(new_incoming_tickets)
probabilities = nlp_router_pipeline.predict_proba(new_incoming_tickets)

print("--- Automated Ticket Routing Predictions ---")
for ticket, dept in zip(new_incoming_tickets, predictions):
    print(f"Ticket: '{ticket}'")
    print(f"  -> Routed To Department: {dept.upper()}\n")

--- Automated Ticket Routing Predictions ---
Ticket: 'I was charged twice on my credit card statement!'
  -> Routed To Department: BILLING

Ticket: 'The application keeps shutting down on Windows 11'
  -> Routed To Department: GENERAL



---
## Hands-on Workshop: NLP & Embeddings

Complete these four practical challenges to build your natural language processing capabilities.

### Challenge 1: Text Tokenization & Cleaning Pipeline (Junior Level)

**Scenario:** Clean raw customer reviews before sentiment modeling.

**Task Requirements:**
1. Given raw review `text = "The product quality was TERRIBLE!!! Worst purchase of 2026."`.
2. Convert to lowercase.
3. Remove punctuation using `re.sub(r"[^\w\s]", "", text)`.
4. Tokenize by splitting on whitespace.
5. Print clean tokens list.

In [5]:
# --- CHALLENGE 1: YOUR WORKSPACE ---


# --- CHALLENGE 1: INSTRUCTOR SOLUTION ---
import re

text = "The product quality was TERRIBLE!!! Worst purchase of 2026."

clean_lower = text.lower()
no_punc = re.sub(r"[^\w\s]", "", clean_lower)
tokens = no_punc.split()

print("Clean Tokens List:", tokens)

Clean Tokens List: ['the', 'product', 'quality', 'was', 'terrible', 'worst', 'purchase', 'of', '2026']


### Challenge 2: TF-IDF Keyword Extraction (Mid Level)

**Scenario:** Extract top TF-IDF keywords from a customer complaint ticket.

**Task Requirements:**
1. Given corpus `["System server connection error", "Billing charge failed on credit card"]`.
2. Fit `TfidfVectorizer()`.
3. Transform corpus into TF-IDF matrix.
4. Print feature names and matrix shape.

In [7]:
# --- CHALLENGE 2: YOUR WORKSPACE ---


# --- CHALLENGE 2: INSTRUCTOR SOLUTION ---
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = ["System server connection error", "Billing charge failed on credit card"]

tfidf = TfidfVectorizer()
matrix = tfidf.fit_transform(corpus)

print("TF-IDF Matrix Shape:", matrix.shape)
print("Extracted Feature Words:", tfidf.get_feature_names_out())

TF-IDF Matrix Shape: (2, 10)
Extracted Feature Words: ['billing' 'card' 'charge' 'connection' 'credit' 'error' 'failed' 'on'
 'server' 'system']


### Challenge 3: Cosine Similarity on Word Vectors (Senior Level)

**Scenario:** Calculate semantic similarity between two word embedding vectors.

**Task Requirements:**
1. Given vector A `[0.8, 0.2, 0.1]` and vector B `[0.7, 0.3, 0.0]`.
2. Compute dot product using `np.dot(a, b)`.
3. Divide by product of L2 norms `(np.linalg.norm(a) * np.linalg.norm(b))`.
4. Print cosine similarity score.

In [8]:
# --- CHALLENGE 3: YOUR WORKSPACE ---


# --- CHALLENGE 3: INSTRUCTOR SOLUTION ---
import numpy as np

vec_a = np.array([0.8, 0.2, 0.1])
vec_b = np.array([0.7, 0.3, 0.0])

dot_prod = np.dot(vec_a, vec_b)
norm_a = np.linalg.norm(vec_a)
norm_b = np.linalg.norm(vec_b)

cosine_sim = dot_prod / (norm_a * norm_b)
print(f"Cosine Similarity Score: {cosine_sim:.4f}")

Cosine Similarity Score: 0.9801


### Challenge 4: Regex Entity Extractor for Order IDs (Architect Level)

**Scenario:** Extract order ID numbers from unstructured email text for automated tracking.

**Task Requirements:**
1. Given email `text = "Hi, checking status for order #10982 and order #55432."`.
2. Write a regex pattern matching order IDs (`r"#\d+"`).
3. Use `re.findall()` to extract all order IDs.
4. Print extracted list.

In [9]:
# --- CHALLENGE 4: YOUR WORKSPACE ---


# --- CHALLENGE 4: INSTRUCTOR SOLUTION ---
import re

email_text = "Hi, checking status for order #10982 and order #55432."

order_pattern = r"#\d+"
extracted_orders = re.findall(order_pattern, email_text)

print("Extracted Order IDs:", extracted_orders)

Extracted Order IDs: ['#10982', '#55432']


# Session Summary

Congratulations. You have completed Notebook 21.

Let us review what you now understand:
1. **Text Preprocessing:** You know how tokenization, stop-word removal, lowercasing, and lemmatization reduce vocabulary noise.
2. **TF-IDF & Topic Modeling:** You understand how TF-IDF penalizes common words and rewards informative keywords, and how LDA discovers unsupervised topics.
3. **Word Embeddings:** You know how Word2Vec, GloVe, and FastText map words to dense semantic vectors where distance represents meaning.
4. **Structural Understanding & Routing:** You can extract named entities (dates, money, order IDs) using regex/NER and build automated text classification routing pipelines.

In **Notebook 22**, we enter **Modern NLP, Transformers & Hugging Face**—learning Self-Attention mechanisms, Transformer architectures (BERT, GPT, T5), and fine-tuning pre-trained LLMs using the Hugging Face library!